# 07 — Core ICNN Lyapunov Experiments

This notebook is the **main project line**.

Goal: compare an unconstrained autoregressive dynamics model against an ICNN/Lyapunov-stable dynamics model on low-dimensional physical systems.

Systems:
1. `mass_spring_damper` — stable 2D system, clean verification case.
2. `damped_pendulum_4` — 8D nonlinear stable system, main experiment with rollout-augmented Lyapunov training.

Metrics:
- derivative MSE,
- rollout error over horizon,
- final/mean/max rollout error,
- Lyapunov decrease satisfaction,
- projection fire rate,
- correction magnitude,
- example trajectories.

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import TensorDataset

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model, make_system
from stable_icnn_physics.data import (
    dataset_base_name,
    generate_derivative_data,
    load_dataset,
    save_dataset,
    tensor_dataset,
)
from stable_icnn_physics.eval import (
    autoregressive_rollout_model,
    lyapunov_decrease_values,
    rollout_error,
    rollout_system,
)
from stable_icnn_physics.train import evaluate_derivative_mse, train_derivative_model

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CACHE_DIR = REPO_ROOT / "data" / "cache"
CKPT_DIR = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "core_icnn_plots"

for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TOLERANCE = 1e-5
SEED = 0

print("repo:", REPO_ROOT)
print("device:", DEVICE, "| torch:", torch.__version__)

## Experiment configuration

The notebook can either train models or load existing checkpoints.

For the 4-link pendulum, default behavior is to reuse your strongest existing result:

```text
stable:   checkpoints/p4_rollout_aug_stable.pt
baseline: checkpoints/p4_random_large_e500_alpha1e5_baseline.pt
```

That keeps this notebook focused on **analysis and final plots**, while still allowing retraining if needed.

In [ ]:
@dataclass
class CoreExperimentConfig:
    tag: str
    system_name: str
    system_kwargs: dict
    dt: float
    steps: int
    train_samples: int
    test_samples: int
    eval_trajs: int
    hidden: int
    depth: int
    lyapunov_hidden: int
    lyapunov_eps: float
    alpha: float
    rehu_width: float = 0.01
    epochs: int = 150
    batch_size: int = 256
    lr: float = 1e-3
    prefer_existing: bool = True
    stable_ckpt_override: str | None = None
    baseline_ckpt_override: str | None = None

EXPERIMENTS = [
    CoreExperimentConfig(
        tag="core_msd_icnn",
        system_name="mass_spring_damper",
        system_kwargs={},
        dt=0.05,
        steps=300,
        train_samples=20_000,
        test_samples=5_000,
        eval_trajs=32,
        hidden=100,
        depth=2,
        lyapunov_hidden=60,
        lyapunov_eps=0.01,
        alpha=1e-3,
        epochs=150,
    ),
    CoreExperimentConfig(
        tag="core_p4_rollout_aug_icnn",
        system_name="damped_pendulum_4",
        system_kwargs={"friction": 0.3, "gravity": 9.81},
        dt=0.02,
        steps=300,
        train_samples=50_000,
        test_samples=10_000,
        eval_trajs=16,
        hidden=200,
        depth=3,
        lyapunov_hidden=100,
        lyapunov_eps=0.01,
        alpha=1e-5,
        epochs=200,
        prefer_existing=True,
        stable_ckpt_override="p4_rollout_aug_stable.pt",
        baseline_ckpt_override="p4_random_large_e500_alpha1e5_baseline.pt",
    ),
]

## Utility functions

In [ ]:
def ensure_random_derivative_dataset(system, train_samples: int, test_samples: int, seed: int = 0):
    paths = {}
    for split, n in [("train", train_samples), ("test", test_samples)]:
        path = CACHE_DIR / dataset_base_name(
            system, split=split, n_samples=n, seed=seed, dataset_type="derivative"
        )
        if not path.exists():
            print(f"generating {split}: {path.name}")
            x, y = generate_derivative_data(system, n_samples=n, split=split, seed=seed)
            save_dataset(path, x, y, metadata={"system": system.metadata(), "split": split})
        else:
            print(f"reusing {split}: {path.name}")
        paths[split] = path

    x_train, y_train = load_dataset(paths["train"])
    x_test, y_test = load_dataset(paths["test"])
    return x_train, y_train, x_test, y_test

def make_stable_for_cfg(cfg: CoreExperimentConfig, state_dim: int):
    return build_stable_model(
        dim=state_dim,
        hidden=cfg.hidden,
        depth=cfg.depth,
        lyapunov_hidden=cfg.lyapunov_hidden,
        lyapunov_eps=cfg.lyapunov_eps,
        alpha=cfg.alpha,
        rehu_width=cfg.rehu_width,
    )

def make_baseline_for_cfg(cfg: CoreExperimentConfig, state_dim: int):
    return BaselineDynamicsMLP(dim=state_dim, hidden=cfg.hidden, depth=cfg.depth)

def load_or_train_models(cfg: CoreExperimentConfig, system, train_ds, test_ds):
    state_dim = system.state_dim

    stable_ckpt = CKPT_DIR / (cfg.stable_ckpt_override or f"{cfg.tag}_stable.pt")
    baseline_ckpt = CKPT_DIR / (cfg.baseline_ckpt_override or f"{cfg.tag}_baseline.pt")

    stable = make_stable_for_cfg(cfg, state_dim)
    baseline = make_baseline_for_cfg(cfg, state_dim)

    can_load = cfg.prefer_existing and stable_ckpt.exists() and baseline_ckpt.exists()
    if can_load:
        print("loading stable:", stable_ckpt)
        stable.load_state_dict(torch.load(stable_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
        print("loading baseline:", baseline_ckpt)
        baseline.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
    else:
        print("training stable ICNN model ->", stable_ckpt)
        train_derivative_model(
            stable, train_ds, test_ds,
            epochs=cfg.epochs,
            batch_size=cfg.batch_size,
            learning_rate=cfg.lr,
            device=DEVICE,
            checkpoint_path=stable_ckpt,
            print_every=max(1, cfg.epochs // 10),
            use_amp=False,
        )
        print("training baseline model ->", baseline_ckpt)
        train_derivative_model(
            baseline, train_ds, test_ds,
            epochs=cfg.epochs,
            batch_size=cfg.batch_size,
            learning_rate=cfg.lr,
            device=DEVICE,
            checkpoint_path=baseline_ckpt,
            print_every=max(1, cfg.epochs // 10),
            use_amp=(DEVICE == "cuda"),
        )

    stable.to(DEVICE).eval()
    baseline.to(DEVICE).eval()
    return stable, baseline, stable_ckpt, baseline_ckpt

def projection_diagnostics(model, traj: np.ndarray):
    fires = []
    correction_norms = []
    violations = []
    v_values = []

    with torch.enable_grad():
        for t in range(traj.shape[0] - 1):
            x = torch.tensor(traj[t], dtype=torch.float32, device=DEVICE).requires_grad_(True)
            if x.dim() == 1:
                x = x.unsqueeze(0)

            fx_hat = model.fhat(x)
            vx = model.V(x)
            grad_v = torch.autograd.grad(vx.sum(), x, create_graph=False)[0]
            vio = (grad_v * fx_hat).sum(dim=1, keepdim=True) + model.alpha * vx
            denom = grad_v.square().sum(dim=1, keepdim=True).clamp_min(model.denom_eps)
            correction = grad_v * (torch.relu(vio) / denom)

            fires.append((vio > 0).detach().cpu().numpy().reshape(-1))
            correction_norms.append(correction.norm(dim=1).detach().cpu().numpy().reshape(-1))
            violations.append(vio.detach().cpu().numpy().reshape(-1))
            v_values.append(vx.detach().cpu().numpy().reshape(-1))

        x_final = torch.tensor(traj[-1], dtype=torch.float32, device=DEVICE).requires_grad_(True)
        if x_final.dim() == 1:
            x_final = x_final.unsqueeze(0)
        v_final = model.V(x_final).detach().cpu().numpy().reshape(-1)
        v_values.append(v_final)

    fire = np.asarray(fires)
    correction_norm = np.asarray(correction_norms)
    violation = np.asarray(violations)
    V = np.asarray(v_values)
    dV = V[1:] - V[:-1]

    return {
        "fire": fire,
        "correction_norm": correction_norm,
        "violation": violation,
        "V": V,
        "dV": dV,
        "projection_fire_rate": float(fire.mean()),
        "correction_norm_mean": float(correction_norm.mean()),
        "correction_norm_p95": float(np.quantile(correction_norm, 0.95)),
        "correction_norm_max": float(correction_norm.max()),
        "nominal_violation_mean": float(violation.mean()),
        "nominal_violation_max": float(violation.max()),
        "discrete_dV_frac_nonpositive": float(np.mean(dV <= TOLERANCE)),
        "discrete_dV_max": float(dV.max()),
    }

def summarize_error(err):
    return {
        "final": float(err[-1]),
        "mean": float(err.mean()),
        "max": float(err.max()),
    }

## Run/evaluate all core experiments

In [ ]:
all_results = {}
all_curves = {}

for cfg in EXPERIMENTS:
    print("\n" + "=" * 80)
    print(cfg.tag)
    print("=" * 80)

    system = make_system(cfg.system_name, **cfg.system_kwargs)
    x_train, y_train, x_test, y_test = ensure_random_derivative_dataset(
        system, cfg.train_samples, cfg.test_samples, seed=SEED
    )
    train_ds = tensor_dataset(x_train, y_train)
    test_ds = tensor_dataset(x_test, y_test)

    x0 = system.sample_initial_conditions(cfg.eval_trajs, split="test", seed=SEED + 123)
    true_traj = rollout_system(system, x0, steps=cfg.steps, dt=cfg.dt)

    stable, baseline, stable_ckpt, baseline_ckpt = load_or_train_models(cfg, system, train_ds, test_ds)

    dmse_stable = evaluate_derivative_mse(stable, test_ds, device=DEVICE)
    dmse_baseline = evaluate_derivative_mse(baseline, test_ds, device=DEVICE)

    stable_traj = autoregressive_rollout_model(
        stable, x0, steps=cfg.steps, dt=cfg.dt, device=DEVICE, wrap_fn=system.wrap_state
    )
    baseline_traj = autoregressive_rollout_model(
        baseline, x0, steps=cfg.steps, dt=cfg.dt, device=DEVICE, wrap_fn=system.wrap_state
    )

    err_stable = rollout_error(system, true_traj, stable_traj).mean(axis=1)
    err_baseline = rollout_error(system, true_traj, baseline_traj).mean(axis=1)

    decrease = lyapunov_decrease_values(stable, x_test[:2048], device=DEVICE).ravel()
    diag = projection_diagnostics(stable, stable_traj)

    result = {
        "config": asdict(cfg),
        "stable_ckpt": str(stable_ckpt.relative_to(REPO_ROOT)) if stable_ckpt.is_relative_to(REPO_ROOT) else str(stable_ckpt),
        "baseline_ckpt": str(baseline_ckpt.relative_to(REPO_ROOT)) if baseline_ckpt.is_relative_to(REPO_ROOT) else str(baseline_ckpt),
        "derivative_mse_stable": float(dmse_stable),
        "derivative_mse_baseline": float(dmse_baseline),
        "rollout_stable": summarize_error(err_stable),
        "rollout_baseline": summarize_error(err_baseline),
        "lyapunov_max_violation_projected": float(decrease.max()),
        "lyapunov_fraction_satisfied_projected": float(np.mean(decrease <= TOLERANCE)),
        "projection_fire_rate": diag["projection_fire_rate"],
        "correction_norm_mean": diag["correction_norm_mean"],
        "correction_norm_p95": diag["correction_norm_p95"],
        "correction_norm_max": diag["correction_norm_max"],
        "nominal_violation_mean": diag["nominal_violation_mean"],
        "nominal_violation_max": diag["nominal_violation_max"],
        "discrete_dV_frac_nonpositive": diag["discrete_dV_frac_nonpositive"],
        "discrete_dV_max": diag["discrete_dV_max"],
    }

    all_results[cfg.tag] = result
    all_curves[cfg.tag] = {
        "t": np.arange(cfg.steps + 1) * cfg.dt,
        "err_stable": err_stable,
        "err_baseline": err_baseline,
        "true_traj": true_traj,
        "stable_traj": stable_traj,
        "baseline_traj": baseline_traj,
        "diag": diag,
        "system": system,
        "cfg": cfg,
    }

    out_path = RESULTS_DIR / f"{cfg.tag}_core_summary.json"
    out_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    print("saved:", out_path)

combined_path = RESULTS_DIR / "core_icnn_experiments_summary.json"
combined_path.write_text(json.dumps(all_results, indent=2), encoding="utf-8")
print("\ncombined summary:", combined_path)

## Summary table

In [ ]:
try:
    import pandas as pd

    rows = []
    for tag, r in all_results.items():
        rows.append({
            "tag": tag,
            "system": r["config"]["system_name"],
            "dmse_stable": r["derivative_mse_stable"],
            "dmse_baseline": r["derivative_mse_baseline"],
            "final_rollout_stable": r["rollout_stable"]["final"],
            "final_rollout_baseline": r["rollout_baseline"]["final"],
            "mean_rollout_stable": r["rollout_stable"]["mean"],
            "mean_rollout_baseline": r["rollout_baseline"]["mean"],
            "lyap_frac": r["lyapunov_fraction_satisfied_projected"],
            "fire_rate": r["projection_fire_rate"],
            "corr_mean": r["correction_norm_mean"],
            "dV_frac_nonpos": r["discrete_dV_frac_nonpositive"],
        })
    df = pd.DataFrame(rows)
    display(df)
    csv_path = RESULTS_DIR / "core_icnn_experiments_summary.csv"
    df.to_csv(csv_path, index=False)
    print("saved:", csv_path)
except Exception as e:
    print("pandas display failed:", e)
    print(json.dumps(all_results, indent=2))

## Plot: rollout error

In [ ]:
for tag, curves in all_curves.items():
    t = curves["t"]
    cfg = curves["cfg"]

    plt.figure(figsize=(8, 4))
    plt.plot(t, curves["err_baseline"], label="baseline")
    plt.plot(t, curves["err_stable"], label="ICNN stable")
    plt.xlabel("time [s]")
    plt.ylabel("mean state error")
    plt.title(f"Rollout error — {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_rollout_error.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

    plt.figure(figsize=(8, 4))
    plt.semilogy(t, curves["err_baseline"] + 1e-12, label="baseline")
    plt.semilogy(t, curves["err_stable"] + 1e-12, label="ICNN stable")
    plt.xlabel("time [s]")
    plt.ylabel("mean state error, log scale")
    plt.title(f"Rollout error log-scale — {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_rollout_error_log.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

## Plot: projection diagnostics

In [ ]:
for tag, curves in all_curves.items():
    diag = curves["diag"]
    t_mid = curves["t"][:-1]

    fire_t = diag["fire"].mean(axis=1)
    corr_mean_t = diag["correction_norm"].mean(axis=1)
    corr_p95_t = np.quantile(diag["correction_norm"], 0.95, axis=1)

    plt.figure(figsize=(8, 4))
    plt.plot(t_mid, fire_t)
    plt.xlabel("time [s]")
    plt.ylabel("fraction of eval trajectories")
    plt.ylim(-0.02, 1.02)
    plt.title(f"Projection fire rate — {tag}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_projection_fire_rate.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

    plt.figure(figsize=(8, 4))
    plt.plot(t_mid, corr_mean_t, label="mean")
    plt.plot(t_mid, corr_p95_t, label="p95")
    plt.xlabel("time [s]")
    plt.ylabel("|| correction ||")
    plt.title(f"Projection correction magnitude — {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_correction_norm.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

    plt.figure(figsize=(8, 4))
    plt.hist(diag["violation"].ravel(), bins=80)
    plt.axvline(0.0, linestyle="--")
    plt.xlabel("nominal violation")
    plt.ylabel("count")
    plt.title(f"Nominal Lyapunov violation distribution — {tag}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_nominal_violation_hist.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

## Plot: Lyapunov value and discrete change

In [ ]:
for tag, curves in all_curves.items():
    diag = curves["diag"]
    t = curves["t"]
    t_mid = t[:-1]

    V_mean = diag["V"].mean(axis=1)
    V_p95 = np.quantile(diag["V"], 0.95, axis=1)
    dV_mean = diag["dV"].mean(axis=1)
    dV_p95 = np.quantile(diag["dV"], 0.95, axis=1)
    dV_max = diag["dV"].max(axis=1)

    plt.figure(figsize=(8, 4))
    plt.plot(t, V_mean, label="mean V")
    plt.plot(t, V_p95, label="p95 V")
    plt.xlabel("time [s]")
    plt.ylabel("V(x)")
    plt.title(f"ICNN Lyapunov value along rollout — {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_lyapunov_value.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

    plt.figure(figsize=(8, 4))
    plt.plot(t_mid, dV_mean, label="mean ΔV")
    plt.plot(t_mid, dV_p95, label="p95 ΔV")
    plt.plot(t_mid, dV_max, label="max ΔV")
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("time [s]")
    plt.ylabel("V(x[t+1]) - V(x[t])")
    plt.title(f"Discrete Lyapunov change — {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"{tag}_discrete_deltaV.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)

## Plot: example state trajectories

In [ ]:
for tag, curves in all_curves.items():
    system = curves["system"]
    cfg = curves["cfg"]
    t = curves["t"]
    traj_id = 0

    dim = curves["true_traj"].shape[-1]
    if cfg.system_name == "mass_spring_damper":
        coord_names = ["position", "velocity"]
    elif cfg.system_name == "damped_pendulum_4":
        coord_names = [f"theta_{i}" for i in range(4)] + [f"omega_{i}" for i in range(4)]
    else:
        coord_names = [f"x{i}" for i in range(dim)]

    for j in range(dim):
        plt.figure(figsize=(8, 3.5))
        plt.plot(t, curves["true_traj"][:, traj_id, j], label="true")
        plt.plot(t, curves["baseline_traj"][:, traj_id, j], label="baseline")
        plt.plot(t, curves["stable_traj"][:, traj_id, j], label="ICNN stable")
        plt.xlabel("time [s]")
        plt.ylabel(coord_names[j])
        plt.title(f"{tag} — {coord_names[j]}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        path = PLOTS_DIR / f"{tag}_traj_{j}_{coord_names[j]}.png"
        plt.savefig(path, dpi=160)
        plt.show()

## Interpretation checklist

For the final report, focus on:

1. Does the baseline have low one-step error but worse long-horizon rollout?
2. Does the stable ICNN model keep trajectories bounded?
3. How often does the Lyapunov projection fire?
4. Is the correction magnitude small enough to avoid destroying the learned nominal dynamics?
5. Does rollout-augmented Lyapunov training improve the pendulum result relative to earlier Exp5/Exp6?

The key story should stay centered on ICNN positive definite `V(x)` and autoregressive stability.